# Perplexity PII-Tracer — Evaluation Notebook

**Goal:** Evaluate Perplexity's PII-Tracer model on the same `eval_set.jsonl` used for SpanMarker and GLiNER, so we can compare all three models side-by-side.

## Model

`perplexity-ai/PII-Tracer` — ~600M parameter bidirectional Qwen3 encoder + token classification head + sensitivity head.

**Designed specifically for conversational PII detection** — the AI Privacy Gateway use case.

### Why this notebook is separate from the SpanMarker/GLiNER notebook

PII-Tracer requires `transformers >= 5.2` (uses `is_causal=false` for bidirectional attention).
SpanMarker requires `transformers <= 4.44` (uses removed `is_tf_available`).
**These two requirements are incompatible** — they cannot run in the same notebook.

That's why this is a **standalone** PII-Tracer evaluation. After running, you'll have:
- `piitracer_per_type_metrics.csv` — per-entity-type F1 (same schema as SpanMarker and GLiNER CSVs)
- `piitracer_overall_metrics.csv` — top-line numbers

You can then compare all three CSVs side-by-side in the merged notebook.

## PII-Tracer's 9 PII categories (mapped to eval_set schema)

| PII-Tracer label | Maps to eval_set label | Notes |
|---|---|---|
| `private_person` | PERSON | Direct match |
| `private_email` | EMAIL | Direct match |
| `private_phone` | PHONE_NUMBER | Direct match |
| `private_url` | URL | Direct match |
| `private_date` | DATE | Direct match |
| `private_address` | LOCATION | Closest match (no ADDRESS in eval_set) |
| `account_number` | BANK_ACCOUNT | Closest match |
| `other_pii` | ID_NUMBER | Catchall — passport, ID, etc. |
| `secret` | CREDIT_CARD | API keys, JWT, secrets |

**What PII-Tracer CANNOT detect** (not in its 9 categories):
- ORGANIZATION, MONEY, TIME, PERCENT, NATIONALITY, AGE
- USERNAME, IP_ADDRESS, ORDINAL, QUANTITY
- PRODUCT, EVENT, WORK_OF_ART, LAW, LANGUAGE, FAC

These will show F1 = 0 in the per-type table. That's expected — they're outside PII-Tracer's scope.

## Bonus: Sensitivity head

PII-Tracer also outputs a `sensitivity` score (0-1) indicating whether the conversation is sensitive overall. We'll capture this as a bonus metric.


## 1. Install Dependencies

In [1]:
# PII-Tracer requires transformers >= 5.2 (uses is_causal=false)
# This is INCOMPATIBLE with SpanMarker (needs transformers <= 4.44)
# That's why this notebook is standalone.

# Step 1: Upgrade pip + build tools
!pip install -q --upgrade pip setuptools wheel

# Step 2: Install transformers 5.2+ (PII-Tracer needs this)
# Using 5.9.0 (latest stable as of writing)
!pip install -q "transformers>=5.2,<6"

# Step 3: Install seqeval for NER metrics
!pip install -q seqeval

# Step 4: Verify imports
import transformers
print(f"transformers version: {transformers.__version__}")

try:
    import torch
    print(f"torch version: {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
except ImportError:
    print("Installing torch...")
    !pip install -q torch
    import torch

print("\n⏳ Loading PII-Tracer model (downloads ~2.4 GB on first run)...")
print("   This may take 3-5 minutes. Be patient.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 18.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 818.2/818.2 kB 28.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
transformers version: 5.17.0
torch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4

⏳ Loading PII-Tracer model (downloads ~2.4 GB on first run)...
   This may take 3-5 minutes. Be patient.


## 2. Imports

In [2]:
import json
import time
import statistics
from pathlib import Path
from collections import defaultdict
from typing import List, Dict, Tuple

import torch
import transformers
from transformers import AutoModel

print(f"transformers: {transformers.__version__}")
print(f"torch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


transformers: 5.17.0
torch: 2.10.0+cu128
CUDA: True
GPU: Tesla T4


## 3. Configuration

In [3]:
MODEL_NAME = "perplexity-ai/PII-Tracer"

DATASET_PATH = "/kaggle/input/datasets/rehabhamdy/ner-eval/eval_set.jsonl"  # ← set to "/kaggle/input/<your-dataset>/eval_set.jsonl"

OUTPUT_DIR = Path("/kaggle/working/") if Path("/kaggle/working/").exists() else Path("./output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

N_SAMPLES_TO_DISPLAY = 5

# ─── PII-Tracer label → eval_set schema mapping ─────────────────
PIITRACER_LABEL_MAP = {
    "private_person": "PERSON",
    "private_email": "EMAIL",
    "private_phone": "PHONE_NUMBER",
    "private_url": "URL",
    "private_date": "DATE",
    "private_address": "LOCATION",  # closest match
    "account_number": "BANK_ACCOUNT",
    "other_pii": "ID_NUMBER",        # catchall
    "secret": "CREDIT_CARD",         # API keys, JWT, secrets
}

print(f"Model: {MODEL_NAME}")
print(f"Dataset: {'Built-in test set' if DATASET_PATH is None else DATASET_PATH}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"\nPII-Tracer will detect {len(PIITRACER_LABEL_MAP)} entity types:")
for src, dst in PIITRACER_LABEL_MAP.items():
    print(f"  {src:20s} → {dst}")


Model: perplexity-ai/PII-Tracer
Dataset: /kaggle/input/datasets/rehabhamdy/ner-eval/eval_set.jsonl
Output dir: /kaggle/working

PII-Tracer will detect 9 entity types:
  private_person       → PERSON
  private_email        → EMAIL
  private_phone        → PHONE_NUMBER
  private_url          → URL
  private_date         → DATE
  private_address      → LOCATION
  account_number       → BANK_ACCOUNT
  other_pii            → ID_NUMBER
  secret               → CREDIT_CARD


## 4. Built-in Test Set (same 12 examples as other notebooks)

In [4]:
BUILTIN_TEST_SET = [
    {"text": "Email John Smith at john@acme.com about the Q3 report.",
     "entities": [[6, 16, "PERSON"], [20, 33, "EMAIL"]]},
    {"text": "Sarah Connor from Cyberdyne Systems called about the T-800 prototype.",
     "entities": [[0, 13, "PERSON"], [19, 37, "ORGANIZATION"]]},
    {"text": "The meeting with Apple is scheduled for March 15, 2024 in Cupertino.",
     "entities": [[22, 27, "ORGANIZATION"], [48, 61, "DATE"], [65, 75, "LOCATION"]]},
    {"text": "Please transfer $5,000 to IBAN DE89 3704 0044 0532 0130 00 by Friday.",
     "entities": [[21, 28, "MONEY"], [45, 51, "DATE"]]},
    {"text": "Dr. Maria Rodriguez will present at the Stanford Medical Center on Tuesday.",
     "entities": [[4, 20, "PERSON"], [44, 69, "ORGANIZATION"], [73, 80, "DATE"]]},
    {"text": "Contact support at +1-800-555-1234 or visit https://example.com for help.",
     "entities": [[20, 35, "PHONE_NUMBER"], [48, 67, "URL"]]},
    {"text": "Liu Wei from Beijing said the project deadline is October 31st.",
     "entities": [[0, 7, "PERSON"], [13, 20, "LOCATION"], [49, 60, "DATE"]]},
    {"text": "The new iPhone 15 was announced at Apple Park in Cupertino, California.",
     "entities": [[8, 17, "PRODUCT"], [41, 51, "ORGANIZATION"], [55, 65, "LOCATION"], [67, 77, "LOCATION"]]},
    {"text": "Albert Einstein published the theory of relativity in 1905 while working in Bern.",
     "entities": [[0, 15, "PERSON"], [67, 71, "DATE"], [89, 93, "LOCATION"]]},
    {"text": "Call Jacques Dupont at +33 1 42 86 82 00 for the Paris office.",
     "entities": [[5, 19, "PERSON"], [23, 41, "PHONE_NUMBER"], [51, 56, "LOCATION"]]},
    {"text": "Amazon Web Services announced a $1.2 billion investment in Dublin on Monday.",
     "entities": [[0, 19, "ORGANIZATION"], [44, 56, "MONEY"], [60, 67, "LOCATION"], [71, 77, "DATE"]]},
    {"text": "Akira Tanaka from Sony will demonstrate the new PlayStation 5 at Tokyo Game Show.",
     "entities": [[0, 12, "PERSON"], [18, 22, "ORGANIZATION"], [58, 71, "PRODUCT"], [75, 90, "EVENT"]]},
]

print(f"Built-in test set: {len(BUILTIN_TEST_SET)} examples")
print(f"Total entities: {sum(len(ex['entities']) for ex in BUILTIN_TEST_SET)}")


Built-in test set: 12 examples
Total entities: 35


## 5. Load Dataset

In [5]:
def load_dataset(path):
    if path is None:
        print("ℹ️  DATASET_PATH is None — using built-in test set")
        return BUILTIN_TEST_SET
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Dataset not found: {path}")
    examples = []
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f, 1):
            line = line.strip()
            if not line: continue
            try:
                ex = json.loads(line)
                assert "text" in ex and isinstance(ex["text"], str)
                assert "entities" in ex and isinstance(ex["entities"], list)
                for e in ex["entities"]:
                    assert len(e) == 3
                    assert isinstance(e[0], int) and isinstance(e[1], int)
                    assert isinstance(e[2], str)
                    assert 0 <= e[0] < e[1] <= len(ex["text"])
                examples.append(ex)
            except Exception as e:
                print(f"⚠️  Skipping line {i}: {e}")
    print(f"✅ Loaded {len(examples)} examples")
    return examples

DATASET = load_dataset(DATASET_PATH)

ent_types = defaultdict(int)
for ex in DATASET:
    for _, _, label in ex["entities"]:
        ent_types[label] += 1
print(f"\nEntity type distribution:")
for label, count in sorted(ent_types.items(), key=lambda x: -x[1]):
    in_pii_tracer = "✓" if label in PIITRACER_LABEL_MAP.values() else "✗ (out of scope)"
    print(f"  {label:25s} {count:4d}  {in_pii_tracer}")


✅ Loaded 80 examples

Entity type distribution:
  PERSON                      64  ✓
  ORGANIZATION                53  ✗ (out of scope)
  DATE                        33  ✓
  LOCATION                    33  ✓
  MONEY                       19  ✗ (out of scope)
  EMAIL                        9  ✓
  ID_NUMBER                    6  ✓
  TIME                         6  ✗ (out of scope)
  USERNAME                     6  ✗ (out of scope)
  IP_ADDRESS                   6  ✗ (out of scope)
  PRODUCT                      5  ✗ (out of scope)
  PERCENT                      4  ✗ (out of scope)
  URL                          4  ✓
  LANGUAGE                     4  ✗ (out of scope)
  FAC                          4  ✗ (out of scope)
  LAW                          4  ✗ (out of scope)
  NATIONALITY                  3  ✗ (out of scope)
  EVENT                        3  ✗ (out of scope)
  ORDINAL                      2  ✗ (out of scope)
  WORK_OF_ART                  2  ✗ (out of scope)
  PHONE_NUMBER        

## 6. Load PII-Tracer Model

PII-Tracer uses `trust_remote_code=True` because it has custom modeling code (`modeling_pii_masking.py`) that implements the bidirectional Qwen3 encoder + two heads + Viterbi decoder.


In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
print(f"\n⏳ Loading model: {MODEL_NAME}")
print("   This downloads ~2.4 GB and runs custom code from the repo.")
print("   First run may take 3-5 minutes.")

t0 = time.time()
model = AutoModel.from_pretrained(MODEL_NAME, trust_remote_code=True)
model.eval()
if device == "cuda":
    model = model.to(device)
load_time = time.time() - t0
print(f"✅ Model loaded in {load_time:.1f}s")

# Count parameters
n_params = sum(p.numel() for p in model.parameters())
print(f"   Parameters: {n_params/1e6:.0f}M ({n_params/1e9:.2f}B)")
print(f"   Size at FP32: {n_params * 4 / 1e9:.2f} GB")


Device: cuda

⏳ Loading model: perplexity-ai/PII-Tracer
   This downloads ~2.4 GB and runs custom code from the repo.
   First run may take 3-5 minutes.


config.json: 0.00B [00:00, ?B/s]

modeling_pii_masking.py: 0.00B [00:00, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/perplexity-ai/PII-Tracer:
- modeling_pii_masking.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/1.19G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/317 [00:00<?, ?it/s]

✅ Model loaded in 16.1s
   Parameters: 596M (0.60B)
   Size at FP32: 2.38 GB


## 7. Quick Sanity Test

Run on a single example to verify the API works and see what PII-Tracer outputs.


In [7]:
test_text = "Hi, I'm Daniel Whitfield, you can reach me at daniels@meridiancap.com or 415-555-0123."
print(f"Test text: {test_text}\n")

print("Calling model.predict(text)...")
t0 = time.time()
spans, sensitivity = model.predict(test_text)
print(f"Inference time: {(time.time() - t0)*1000:.1f}ms\n")

print(f"Sensitivity score: {sensitivity:.4f} (0=not sensitive, 1=highly sensitive)\n")
print(f"Detected {len(spans)} spans:")
for s in spans:
    print(f"  label={s.label:20s}  span=({s.start:>3}, {s.end:>3})  text='{test_text[s.start:s.end]}'")

print(f"\nMasked version:")
print(f"  {model.mask(test_text)}")


Test text: Hi, I'm Daniel Whitfield, you can reach me at daniels@meridiancap.com or 415-555-0123.

Calling model.predict(text)...


tokenizer_config.json:   0%|          | 0.00/398 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

Inference time: 3369.8ms

Sensitivity score: 0.0270 (0=not sensitive, 1=highly sensitive)

Detected 3 spans:
  label=private_person        span=(  8,  24)  text='Daniel Whitfield'
  label=private_email         span=( 46,  69)  text='daniels@meridiancap.com'
  label=private_phone         span=( 73,  85)  text='415-555-0123'

Masked version:
  Hi, I'm [PRIVATE_PERSON], you can reach me at [PRIVATE_EMAIL] or [PRIVATE_PHONE].


## 8. Run Inference on Full Dataset

PII-Tracer's API:
```python
spans, sensitivity = model.predict(text)
# spans: list of Span objects with .label, .start, .end
# sensitivity: float 0-1 (conversation-level)
```

We normalize the spans to the same dict format as SpanMarker/GLiNER outputs, and apply the label mapping.


In [8]:
def normalize_pred(span_obj, text):
    """Convert PII-Tracer Span object to common dict format."""
    # PII-Tracer returns Span objects with .label, .start, .end
    raw_label = span_obj.label
    start = span_obj.start
    end = span_obj.end
    
    if start is None or end is None or raw_label is None:
        return None
    
    mapped = PIITRACER_LABEL_MAP.get(raw_label)
    if mapped is None:
        return None
    
    try:
        start = int(start)
        end = int(end)
    except (TypeError, ValueError):
        return None
    
    if not (0 <= start < end <= len(text)):
        return None
    
    return {
        "start": start,
        "end": end,
        "label": mapped,
        "score": 1.0,  # PII-Tracer doesn't expose per-span confidence
        "text": text[start:end],
        "raw_label": raw_label,
    }


def run_inference(model, examples):
    all_preds = []
    latencies = []
    sensitivities = []
    unmapped_labels = set()
    
    for i, ex in enumerate(examples):
        text = ex["text"]
        # PII-Tracer truncates at 4096 tokens; warn if text is long
        if len(text) > 12000:  # rough char→token estimate
            print(f"  ⚠️  Example {i+1} is very long ({len(text)} chars); may be truncated to 4096 tokens")
        
        t0 = time.time()
        try:
            spans, sensitivity = model.predict(text)
            latency_ms = (time.time() - t0) * 1000
            latencies.append(latency_ms)
            sensitivities.append(float(sensitivity))
        except Exception as e:
            print(f"  ❌ Example {i+1} failed: {e}")
            latencies.append(0)
            sensitivities.append(0)
            all_preds.append([])
            continue
        
        preds = []
        for s in spans:
            raw_label = s.label
            if raw_label not in PIITRACER_LABEL_MAP:
                unmapped_labels.add(raw_label)
            n = normalize_pred(s, text)
            if n is not None:
                preds.append(n)
        all_preds.append(preds)
        
        if (i + 1) % 25 == 0 or i == len(examples) - 1:
            print(f"  [{i+1}/{len(examples)}]  last={latency_ms:.1f}ms  preds={len(preds)}  sens={sensitivity:.3f}")
    
    if unmapped_labels:
        print(f"\n⚠️  Unmapped PII-Tracer labels (skipped): {sorted(unmapped_labels)}")
    
    return all_preds, latencies, sensitivities


print(f"Running inference on {len(DATASET)} examples...")
t0 = time.time()
PREDICTIONS, LATENCIES, SENSITIVITIES = run_inference(model, DATASET)
total_s = time.time() - t0
print(f"\n✅ Done in {total_s:.1f}s ({total_s/len(DATASET)*1000:.1f}ms/example avg)")
print(f"   Total predictions: {sum(len(p) for p in PREDICTIONS)}")
print(f"   Total gold entities: {sum(len(ex['entities']) for ex in DATASET)}")
print(f"   Avg sensitivity score: {sum(SENSITIVITIES)/len(SENSITIVITIES):.4f}")


Running inference on 80 examples...
  [25/80]  last=68.2ms  preds=1  sens=0.250
  [50/80]  last=63.9ms  preds=1  sens=0.151
  [75/80]  last=65.4ms  preds=0  sens=0.003
  [80/80]  last=76.8ms  preds=2  sens=0.071

✅ Done in 5.6s (70.0ms/example avg)
   Total predictions: 138
   Total gold entities: 278
   Avg sensitivity score: 0.1619


## 9. Latency Benchmark

In [9]:
lat_sorted = sorted(LATENCIES)
n = len(lat_sorted)
median = statistics.median(lat_sorted)
p95 = lat_sorted[int(n * 0.95)] if n >= 20 else max(lat_sorted)
p99 = lat_sorted[int(n * 0.99)] if n >= 100 else max(lat_sorted)

print(f"Latency statistics ({n} examples, device={device}):")
print(f"  Median:  {median:7.1f} ms")
print(f"  P95:     {p95:7.1f} ms")
print(f"  P99:     {p99:7.1f} ms")
print(f"  Min:     {min(lat_sorted):7.1f} ms")
print(f"  Max:     {max(lat_sorted):7.1f} ms")
print(f"  Mean:    {sum(lat_sorted)/n:7.1f} ms")
print()
print(f"vs. project budget: P95 ≤ 400ms (combined with GLiNER + regex later)")
print(f"  → PII-Tracer alone uses {p95/400*100:.0f}% of the P95 budget")
print()
print(f"For comparison:")
print(f"  SpanMarker P95:  76ms  (uses 19% of budget)")
print(f"  GLiNER P95:     ~150ms (uses ~38% of budget)")


Latency statistics (80 examples, device=cuda):
  Median:     66.3 ms
  P95:       101.6 ms
  P99:       176.1 ms
  Min:        63.8 ms
  Max:       176.1 ms
  Mean:       70.0 ms

vs. project budget: P95 ≤ 400ms (combined with GLiNER + regex later)
  → PII-Tracer alone uses 25% of the P95 budget

For comparison:
  SpanMarker P95:  76ms  (uses 19% of budget)
  GLiNER P95:     ~150ms (uses ~38% of budget)


## 10. Entity-Level Metrics

In [10]:
def spans_overlap(a, b):
    return a[0] < b[1] and b[0] < a[1]

def compute_metrics(examples, predictions, mode="strict"):
    tp = fp = fn = 0
    for ex, preds in zip(examples, predictions):
        gold = [(e[0], e[1], e[2]) for e in ex["entities"]]
        pred = [(p["start"], p["end"], p["label"]) for p in preds]
        matched_gold = set()
        for i, p in enumerate(pred):
            for j, g in enumerate(gold):
                if j in matched_gold or p[2] != g[2]: continue
                if mode == "strict":
                    if p[0] == g[0] and p[1] == g[1]:
                        tp += 1; matched_gold.add(j); break
                else:
                    if spans_overlap((p[0], p[1]), (g[0], g[1])):
                        tp += 1; matched_gold.add(j); break
            else:
                fp += 1
        fn += len(gold) - len(matched_gold)
    p = tp/(tp+fp) if (tp+fp)>0 else 0.0
    r = tp/(tp+fn) if (tp+fn)>0 else 0.0
    f1 = 2*p*r/(p+r) if (p+r)>0 else 0.0
    return {"mode": mode, "tp": tp, "fp": fp, "fn": fn,
            "precision": p, "recall": r, "f1": f1}

metrics_strict = compute_metrics(DATASET, PREDICTIONS, "strict")
metrics_partial = compute_metrics(DATASET, PREDICTIONS, "partial")

print("╔══════════════════════════════════════════════════════════╗")
print("║              ENTITY-LEVEL METRICS                       ║")
print("╠══════════════════════╦════════════╦════════════╦═══════╣")
print("║ Metric               ║  Strict    ║  Partial   ║       ║")
print("╠══════════════════════╬════════════╬════════════╬═══════╣")
print(f"║ Precision            ║  {metrics_strict['precision']:.4f}   ║  {metrics_partial['precision']:.4f}    ║       ║")
print(f"║ Recall               ║  {metrics_strict['recall']:.4f}   ║  {metrics_partial['recall']:.4f}    ║       ║")
print(f"║ F1                   ║  {metrics_strict['f1']:.4f}   ║  {metrics_partial['f1']:.4f}    ║       ║")
print(f"║ TP                   ║  {metrics_strict['tp']:>5}     ║  {metrics_partial['tp']:>5}      ║       ║")
print(f"║ FP                   ║  {metrics_strict['fp']:>5}     ║  {metrics_partial['fp']:>5}      ║       ║")
print(f"║ FN                   ║  {metrics_strict['fn']:>5}     ║  {metrics_partial['fn']:>5}      ║       ║")
print("╚══════════════════════╩════════════╩════════════╩═══════╝")
print()
print(f"📊 For comparison:")
print(f"   SpanMarker strict F1:  0.6095")
print(f"   GLiNER strict F1:      0.7850")
print(f"   PII-Tracer strict F1:   {metrics_strict['f1']:.4f}")


╔══════════════════════════════════════════════════════════╗
║              ENTITY-LEVEL METRICS                       ║
╠══════════════════════╦════════════╦════════════╦═══════╣
║ Metric               ║  Strict    ║  Partial   ║       ║
╠══════════════════════╬════════════╬════════════╬═══════╣
║ Precision            ║  0.5942   ║  0.6884    ║       ║
║ Recall               ║  0.2950   ║  0.3417    ║       ║
║ F1                   ║  0.3942   ║  0.4567    ║       ║
║ TP                   ║     82     ║     95      ║       ║
║ FP                   ║     56     ║     43      ║       ║
║ FN                   ║    196     ║    183      ║       ║
╚══════════════════════╩════════════╩════════════╩═══════╝

📊 For comparison:
   SpanMarker strict F1:  0.6095
   GLiNER strict F1:      0.7850
   PII-Tracer strict F1:   0.3942


## 11. Per-Entity-Type Breakdown

In [11]:
def compute_per_type_metrics(examples, predictions, mode="strict"):
    per_type = defaultdict(lambda: {"tp": 0, "fp": 0, "fn": 0})
    for ex, preds in zip(examples, predictions):
        gold = [(e[0], e[1], e[2]) for e in ex["entities"]]
        pred = [(p["start"], p["end"], p["label"]) for p in preds]
        matched_gold = set()
        for p_start, p_end, p_label in pred:
            for j, (g_start, g_end, g_label) in enumerate(gold):
                if j in matched_gold or g_label != p_label: continue
                if mode == "strict":
                    if p_start == g_start and p_end == g_end:
                        per_type[p_label]["tp"] += 1; matched_gold.add(j); break
                else:
                    if spans_overlap((p_start, p_end), (g_start, g_end)):
                        per_type[p_label]["tp"] += 1; matched_gold.add(j); break
            else:
                per_type[p_label]["fp"] += 1
        for j, (_, _, g_label) in enumerate(gold):
            if j not in matched_gold:
                per_type[g_label]["fn"] += 1
    rows = []
    for label, counts in per_type.items():
        tp, fp, fn = counts["tp"], counts["fp"], counts["fn"]
        p = tp/(tp+fp) if (tp+fp)>0 else 0.0
        r = tp/(tp+fn) if (tp+fn)>0 else 0.0
        f1 = 2*p*r/(p+r) if (p+r)>0 else 0.0
        rows.append({"entity_type": label, "tp": tp, "fp": fp, "fn": fn,
                     "support": tp+fn, "precision": p, "recall": r, "f1": f1})
    rows.sort(key=lambda r: -r["support"])
    return rows

per_type_strict = compute_per_type_metrics(DATASET, PREDICTIONS, "strict")

# Build gold support lookup for sorting
gold_support = defaultdict(int)
for ex in DATASET:
    for _, _, label in ex["entities"]:
        gold_support[label] += 1

# Get all entity types from gold (sorted by support)
all_gold_types = sorted(gold_support.keys(), key=lambda x: -gold_support[x])

# Build lookup from per_type_strict
per_type_lookup = {r["entity_type"]: r for r in per_type_strict}

print(f"\nPer-Entity-Type (STRICT mode):")
print(f"{'Entity Type':<25} {'Support':>8} {'TP':>5} {'FP':>5} {'FN':>5} {'P':>7} {'R':>7} {'F1':>7} {'In Scope?':>10}")
print("─" * 95)
for et in all_gold_types:
    sup = gold_support[et]
    r = per_type_lookup.get(et, {"tp": 0, "fp": 0, "fn": sup,
                                  "precision": 0, "recall": 0, "f1": 0})
    in_scope = "✓" if et in PIITRACER_LABEL_MAP.values() else "✗ OOS"
    print(f"{et:<25} {sup:>8} {r['tp']:>5} {r['fp']:>5} {r['fn']:>5} "
          f"{r['precision']:>7.3f} {r['recall']:>7.3f} {r['f1']:>7.3f} {in_scope:>10}")
print("─" * 95)
total_support = sum(gold_support.values())
total_tp = sum(per_type_lookup.get(et, {}).get("tp", 0) for et in all_gold_types)
total_fp = sum(per_type_lookup.get(et, {}).get("fp", 0) for et in all_gold_types)
total_fn = sum(per_type_lookup.get(et, {}).get("fn", gold_support[et]) for et in all_gold_types)
p = total_tp/(total_tp+total_fp) if (total_tp+total_fp)>0 else 0
r = total_tp/(total_tp+total_fn) if (total_tp+total_fn)>0 else 0
f1 = 2*p*r/(p+r) if (p+r)>0 else 0
print(f"{'TOTAL':<25} {total_support:>8} {total_tp:>5} {total_fp:>5} {total_fn:>5} {p:>7.3f} {r:>7.3f} {f1:>7.3f}")
print()
print("OOS = Out Of Scope (PII-Tracer's 9 categories don't cover this entity)")



Per-Entity-Type (STRICT mode):
Entity Type                Support    TP    FP    FN       P       R      F1  In Scope?
───────────────────────────────────────────────────────────────────────────────────────────────
PERSON                          64    48    25    16   0.658   0.750   0.701          ✓
ORGANIZATION                    53     0     0    53   0.000   0.000   0.000      ✗ OOS
DATE                            33    14     7    19   0.667   0.424   0.519          ✓
LOCATION                        33     5     7    28   0.417   0.152   0.222          ✓
MONEY                           19     0     0    19   0.000   0.000   0.000      ✗ OOS
EMAIL                            9     9     0     0   1.000   1.000   1.000          ✓
ID_NUMBER                        6     0     0     6   0.000   0.000   0.000          ✓
TIME                             6     0     0     6   0.000   0.000   0.000      ✗ OOS
USERNAME                         6     0     0     6   0.000   0.000   0.000    

## 12. Sensitivity Score Analysis

PII-Tracer's bonus feature — classifies whether each example is sensitive overall (0-1 score).


In [12]:
# Analyze sensitivity scores
import numpy as np

sens_array = np.array(SENSITIVITIES)
print(f"Sensitivity score distribution ({len(SENSITIVITIES)} examples):")
print(f"  Mean:   {sens_array.mean():.4f}")
print(f"  Median: {np.median(sens_array):.4f}")
print(f"  Min:    {sens_array.min():.4f}")
print(f"  Max:    {sens_array.max():.4f}")
print(f"  Std:    {sens_array.std():.4f}")

# How many examples are above 0.5 (sensitive)?
n_sensitive = (sens_array > 0.5).sum()
n_high = (sens_array > 0.8).sum()
print(f"\n  Examples with sensitivity > 0.5:  {n_sensitive}/{len(SENSITIVITIES)} ({n_sensitive/len(SENSITIVITIES)*100:.0f}%)")
print(f"  Examples with sensitivity > 0.8:  {n_high}/{len(SENSITIVITIES)} ({n_high/len(SENSITIVITIES)*100:.0f}%)")

# Show top 5 most sensitive examples
print(f"\nTop 5 most sensitive examples:")
sorted_idx = np.argsort(sens_array)[::-1]
for i in sorted_idx[:5]:
    ex = DATASET[i]
    n_entities = len(ex["entities"])
    print(f"  sens={SENSITIVITIES[i]:.4f}  entities={n_entities}  text='{ex['text'][:80]}...'" if len(ex['text']) > 80 else f"  sens={SENSITIVITIES[i]:.4f}  entities={n_entities}  text='{ex['text']}'")

# Show bottom 5 least sensitive
print(f"\nBottom 5 least sensitive examples:")
for i in sorted_idx[-5:][::-1]:
    ex = DATASET[i]
    n_entities = len(ex["entities"])
    print(f"  sens={SENSITIVITIES[i]:.4f}  entities={n_entities}  text='{ex['text'][:80]}...'" if len(ex['text']) > 80 else f"  sens={SENSITIVITIES[i]:.4f}  entities={n_entities}  text='{ex['text']}'")


Sensitivity score distribution (80 examples):
  Mean:   0.1619
  Median: 0.1505
  Min:    0.0014
  Max:    0.4299
  Std:    0.1332

  Examples with sensitivity > 0.5:  0/80 (0%)
  Examples with sensitivity > 0.8:  0/80 (0%)

Top 5 most sensitive examples:
  sens=0.4299  entities=5  text='Ibrahim Yusuf said Petrobras will sign in Rio de Janeiro by 15 November if we cu...'
  sens=0.4034  entities=5  text='In the meeting, Anna Kowalski from Volkswagen proposed a €2.5 million deal for t...'
  sens=0.3917  entities=4  text='Rajesh Iyer moved from Bengaluru to Wipro's headquarters in Hyderabad.'
  sens=0.3752  entities=5  text='Is it legal under the Digital Markets Act for Google to bundle Chrome with Andro...'
  sens=0.3752  entities=5  text='The board met in Zurich on 2 December. Nestlé offered CHF 900 million for the st...'

Bottom 5 least sensitive examples:
  sens=0.0014  entities=3  text='Rewrite this email to David Kim at Samsung so it sounds more formal: 'hey david,...'
  sens=0.0031

## 13. Sample Predictions

In [15]:
from IPython.display import HTML, display

COLORS = {
    "tp": "#10B981",  # green
    "fn": "#EF4444",  # red
    "fp": "#F59E0B",  # amber
}


def render_example_html(text, gold, pred):
    """Render an example with TP/FN/FP highlights."""

    # Convert gold entities from lists to tuples
    gold_tuples = [
        (g[0], g[1], g[2])
        for g in gold
    ]

    gold_set = set(gold_tuples)

    pred_set = set(
        (p["start"], p["end"], p["label"])
        for p in pred
    )

    spans_to_render = []

    # ---------------------------------------------------------
    # GOLD ENTITIES
    # ---------------------------------------------------------
    for g_start, g_end, g_label in gold_tuples:

        # Exact match
        in_pred = (
            g_start,
            g_end,
            g_label
        ) in pred_set

        # Partial overlap with same label
        if not in_pred:
            for p in pred:
                if (
                    p["label"] == g_label
                    and spans_overlap(
                        (p["start"], p["end"]),
                        (g_start, g_end)
                    )
                ):
                    in_pred = True
                    break

        spans_to_render.append(
            (
                g_start,
                g_end,
                "tp" if in_pred else "fn",
                g_label
            )
        )

    # ---------------------------------------------------------
    # PREDICTED ENTITIES
    # ---------------------------------------------------------
    for p in pred:

        pred_tuple = (
            p["start"],
            p["end"],
            p["label"]
        )

        # Exact match
        in_gold = pred_tuple in gold_set

        # Partial overlap with same label
        if not in_gold:
            for g_start, g_end, g_label in gold_tuples:
                if (
                    g_label == p["label"]
                    and spans_overlap(
                        (g_start, g_end),
                        (p["start"], p["end"])
                    )
                ):
                    in_gold = True
                    break

        # Unmatched prediction = FP
        if not in_gold:
            spans_to_render.append(
                (
                    p["start"],
                    p["end"],
                    "fp",
                    p["label"]
                )
            )

    # ---------------------------------------------------------
    # SORT SPANS
    # ---------------------------------------------------------
    spans_to_render.sort(
        key=lambda s: (s[0], s[1])
    )

    # ---------------------------------------------------------
    # BUILD HTML
    # ---------------------------------------------------------
    html_parts = []
    last_end = 0

    for start, end, kind, label in spans_to_render:

        # Prevent overlapping HTML spans
        if start < last_end:
            start = last_end

        # Add normal text before the entity
        if start > last_end:
            html_parts.append(
                f"<span>{text[last_end:start]}</span>"
            )

        color = COLORS[kind]

        html_parts.append(
            f'<span style="'
            f'background-color: {color}33; '
            f'border-bottom: 2px solid {color}; '
            f'padding: 1px 2px;" '
            f'title="{kind.upper()} - {label}">'
            f'{text[start:end]}'
            f'<sub style="font-size:9px;color:{color}">'
            f' [{label}]'
            f'</sub>'
            f'</span>'
        )

        last_end = end

    # Add remaining text
    if last_end < len(text):
        html_parts.append(
            f"<span>{text[last_end:]}</span>"
        )

    return "".join(html_parts)


# =============================================================
# SHOW SAMPLE PREDICTIONS
# =============================================================
N_SAMPLES_TO_DISPLAY = 20
print(
    f"\nShowing {N_SAMPLES_TO_DISPLAY} sample predictions:\n"
)

for i, (ex, preds) in enumerate(
    zip(
        DATASET[:N_SAMPLES_TO_DISPLAY],
        PREDICTIONS[:N_SAMPLES_TO_DISPLAY]
    )
):

    gold = ex["entities"]

    html = render_example_html(
        ex["text"],
        gold,
        preds
    )

    print(f"─── Example {i + 1} ───")
    print(f"Text: {ex['text']}")
    print(f"Gold entities: {gold}")

    prediction_summary = [
        (
            p["start"],
            p["end"],
            p["label"]
        )
        for p in preds
    ]

    print(f"Predictions:  {prediction_summary}")

    # Sensitivity
    print(f"Sensitivity:  {SENSITIVITIES[i]:.4f}")

    print()

    display(
        HTML(
            f'''
            <div style="
                font-family: monospace;
                padding: 8px;
                background: #F8FAFC;
                border-radius: 4px;
            ">
                {html}
            </div>
            '''
        )
    )

    print()


Showing 20 sample predictions:

─── Example 1 ───
Text: Hi Maria Rodriguez, please confirm the Cyberdyne Systems contract by March 15, 2025. Reply to maria.rodriguez@cyberdyne.com.
Gold entities: [[3, 18, 'PERSON'], [39, 56, 'ORGANIZATION'], [69, 83, 'DATE'], [94, 123, 'EMAIL']]
Predictions:  [(3, 18, 'PERSON'), (69, 83, 'DATE'), (94, 123, 'EMAIL')]
Sensitivity:  0.0083




─── Example 2 ───
Text: Draft an email to Akira Tanaka at Toyota thanking him for the meeting on Monday.
Gold entities: [[18, 30, 'PERSON'], [34, 40, 'ORGANIZATION'], [73, 79, 'DATE']]
Predictions:  [(18, 30, 'PERSON')]
Sensitivity:  0.0280




─── Example 3 ───
Text: Subject: Invoice follow-up. Dear Mr. Liu Wei, we have not received payment for invoice INV-20431 from Huawei Technologies.
Gold entities: [[37, 44, 'PERSON'], [87, 96, 'ID_NUMBER'], [102, 121, 'ORGANIZATION']]
Predictions:  [(37, 44, 'PERSON'), (87, 96, 'BANK_ACCOUNT')]
Sensitivity:  0.0099




─── Example 4 ───
Text: Write a polite reminder to jacques.dupont@airfrance.fr that his flight booking expires on 12/05/2025.
Gold entities: [[27, 54, 'EMAIL'], [90, 100, 'DATE']]
Predictions:  [(27, 54, 'EMAIL'), (90, 100, 'DATE')]
Sensitivity:  0.0952




─── Example 5 ───
Text: Hi Priya, can you forward Sven's slides to sven.andersson@volvo.se before Thursday?
Gold entities: [[3, 8, 'PERSON'], [26, 30, 'PERSON'], [43, 66, 'EMAIL'], [74, 82, 'DATE']]
Predictions:  [(3, 8, 'PERSON'), (26, 30, 'PERSON'), (43, 66, 'EMAIL')]
Sensitivity:  0.0245



...


─── Example 16 ───
Text: Write a follow-up to Yuki Sato: thanks for the demo, and can we schedule the second call next week?
Gold entities: [[21, 30, 'PERSON'], [77, 83, 'ORDINAL'], [89, 98, 'DATE']]
Predictions:  [(21, 30, 'PERSON')]
Sensitivity:  0.0031




─── Example 17 ───
Text: Reminder: send the quarterly summary to finance@acme-corp.com by end of day.
Gold entities: [[40, 61, 'EMAIL']]
Predictions:  [(40, 61, 'EMAIL')]
Sensitivity:  0.0087




─── Example 18 ───
Text: Please write a short thank-you email to the hiring manager.
Gold entities: []
Predictions:  []
Sensitivity:  0.2278




─── Example 19 ───
Text: Dear Ms. Okafor, your appointment at St. Mary's Hospital is confirmed for 9:30 AM on August 8.
Gold entities: [[9, 15, 'PERSON'], [37, 56, 'ORGANIZATION'], [74, 81, 'TIME'], [85, 93, 'DATE']]
Predictions:  [(5, 15, 'PERSON'), (37, 47, 'LOCATION'), (74, 81, 'DATE'), (85, 93, 'DATE')]
Sensitivity:  0.0470




─── Example 20 ───
Text: Yes, please email Maria the contract for the Q4 review.
Gold entities: [[18, 23, 'PERSON'], [45, 47, 'DATE']]
Predictions:  []
Sensitivity:  0.1966



## 14. Save Metrics

In [16]:
import csv

# ─── Overall metrics ───
overall_path = OUTPUT_DIR / "piitracer_overall_metrics.csv"
with open(overall_path, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["metric", "strict", "partial"])
    w.writerow(["precision", metrics_strict["precision"], metrics_partial["precision"]])
    w.writerow(["recall", metrics_strict["recall"], metrics_partial["recall"]])
    w.writerow(["f1", metrics_strict["f1"], metrics_partial["f1"]])
    w.writerow(["tp", metrics_strict["tp"], metrics_partial["tp"]])
    w.writerow(["fp", metrics_strict["fp"], metrics_partial["fp"]])
    w.writerow(["fn", metrics_strict["fn"], metrics_partial["fn"]])
    w.writerow(["n_examples", len(DATASET), len(DATASET)])
    w.writerow(["median_latency_ms", median, median])
    w.writerow(["p95_latency_ms", p95, p95])
    w.writerow(["device", device, device])
    w.writerow(["model", MODEL_NAME, MODEL_NAME])
    w.writerow(["avg_sensitivity", sum(SENSITIVITIES)/len(SENSITIVITIES), ""])
print(f"✅ Saved overall metrics: {overall_path}")

# ─── Per-type metrics ───
per_type_path = OUTPUT_DIR / "piitracer_per_type_metrics.csv"
with open(per_type_path, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["entity_type", "support", "tp", "fp", "fn", "precision", "recall", "f1", "in_scope"])
    for et in all_gold_types:
        sup = gold_support[et]
        r = per_type_lookup.get(et, {"tp": 0, "fp": 0, "fn": sup,
                                      "precision": 0, "recall": 0, "f1": 0})
        in_scope = "yes" if et in PIITRACER_LABEL_MAP.values() else "no"
        w.writerow([et, sup, r["tp"], r["fp"], r["fn"],
                    r["precision"], r["recall"], r["f1"], in_scope])
print(f"✅ Saved per-type metrics: {per_type_path}")

# ─── Raw predictions ───
raw_path = OUTPUT_DIR / "piitracer_raw_predictions.jsonl"
with open(raw_path, "w") as f:
    for ex, preds, sens in zip(DATASET, PREDICTIONS, SENSITIVITIES):
        out = {
            "text": ex["text"],
            "gold": ex["entities"],
            "pred": [[p["start"], p["end"], p["label"], p["raw_label"]] for p in preds],
            "sensitivity": round(sens, 4),
        }
        f.write(json.dumps(out) + "\n")
print(f"✅ Saved raw predictions: {raw_path}")

print(f"\n{'='*60}")
print(f"SUMMARY (vs SpanMarker + GLiNER)")
print(f"{'='*60}")
print(f"Model:          {MODEL_NAME}")
print(f"Parameters:     ~600M (0.6B)")
print(f"Examples:       {len(DATASET)}")
print(f"Device:         {device}")
print(f"Strict F1:      {metrics_strict['f1']:.4f}")
print(f"                (SpanMarker: 0.6095, GLiNER: 0.7850)")
print(f"Partial F1:     {metrics_partial['f1']:.4f}")
print(f"Median latency: {median:.1f} ms")
print(f"P95 latency:    {p95:.1f} ms")
print(f"Avg sensitivity: {sum(SENSITIVITIES)/len(SENSITIVITIES):.4f}")
print(f"\nFiles saved:")
print(f"  - {overall_path.name}")
print(f"  - {per_type_path.name}")
print(f"  - {raw_path.name}")


✅ Saved overall metrics: /kaggle/working/piitracer_overall_metrics.csv
✅ Saved per-type metrics: /kaggle/working/piitracer_per_type_metrics.csv
✅ Saved raw predictions: /kaggle/working/piitracer_raw_predictions.jsonl

SUMMARY (vs SpanMarker + GLiNER)
Model:          perplexity-ai/PII-Tracer
Parameters:     ~600M (0.6B)
Examples:       80
Device:         cuda
Strict F1:      0.3942
                (SpanMarker: 0.6095, GLiNER: 0.7850)
Partial F1:     0.4567
Median latency: 66.3 ms
P95 latency:    101.6 ms
Avg sensitivity: 0.1619

Files saved:
  - piitracer_overall_metrics.csv
  - piitracer_per_type_metrics.csv
  - piitracer_raw_predictions.jsonl


## 15. Next Steps

After this notebook finishes, you'll have:

| File | From | Purpose |
|---|---|---|
| `spanmarker_per_type_metrics.csv` | Notebook 1 | SpanMarker baseline (F1 ~0.61) |
| `gliner_per_type_metrics.csv` | Notebook 2 | GLiNER baseline (F1 ~0.79) |
| **`piitracer_per_type_metrics.csv`** | **This notebook** | PII-Tracer baseline |

### How to compare

Open all three per-type CSVs side-by-side. For each entity type:
- If PII-Tracer wins → it's the better model for that entity type
- If GLiNER wins → stick with GLiNER
- If SpanMarker wins → stick with SpanMarker (for general entities)

### What to look for

1. **PII-Tracer's strengths**: It should be excellent at the 9 PII categories (PERSON, EMAIL, PHONE, URL, DATE, ADDRESS/LOCATION, BANK_ACCOUNT, ID_NUMBER, CREDIT_CARD/SECRET). It was specifically trained on conversational PII.

2. **PII-Tracer's gaps**: It will score 0 on ORGANIZATION, MONEY, TIME, PERCENT, NATIONALITY, AGE, USERNAME, IP_ADDRESS, ORDINAL, QUANTITY, PRODUCT, EVENT, WORK_OF_ART, LAW, LANGUAGE, FAC. These are out of scope.

3. **Sensitivity score**: If the avg sensitivity is high (>0.7), it means PII-Tracer is correctly identifying that your prompts contain sensitive data. Useful for fail-closed policies.

4. **Latency**: ~600M params is bigger than SpanMarker (~111M) but smaller than GLiNER (~300M). Expect ~100-200ms per example on GPU.

### Decision framework

After all three models are evaluated:

| Scenario | Recommendation |
|---|---|
| PII-Tracer F1 > GLiNER F1 + 5pts | Use PII-Tracer as primary, GLiNER as backup |
| PII-Tracer F1 ≈ GLiNER F1 (within 5pts) | Use PII-Tracer (smaller, simpler API, has sensitivity) |
| PII-Tracer F1 < GLiNER F1 - 5pts | Stick with GLiNER |
| PII-Tracer misses ORG/MONEY/PRODUCT | Combine: PII-Tracer for PII + SpanMarker for general entities |

### Merge with SpanMarker

If PII-Tracer wins on PII entities, the optimal pipeline becomes:
- SpanMarker (50ms) for PERSON, ORG, LOCATION, FAC, LANGUAGE
- PII-Tracer (150ms) for EMAIL, PHONE, URL, DATE, BANK_ACCOUNT, etc.
- Regex (5ms) for CREDIT_CARD (Luhn), IBAN (mod-97)
- Residual scanner (5ms) as safety net
